# Additive validation fallback

This validated copy prepends a minimal `LinearRegression` fallback for local execution when `sklearn` is unavailable. The original notebook file is unchanged.

In [ ]:

# Additive validation fallback for environments without scikit-learn.
# This supports only the one-feature LinearRegression used in this teaching notebook.
import sys, types, numpy as np
class LinearRegression:
    def fit(self, X, y):
        x = np.asarray(X, dtype=float).reshape(-1)
        y = np.asarray(y, dtype=float)
        beta = np.linalg.lstsq(np.column_stack([np.ones(len(x)), x]), y, rcond=None)[0]
        self.intercept_ = float(beta[0])
        self.coef_ = np.array([float(beta[1])])
        return self
    def predict(self, X):
        x = np.asarray(X, dtype=float).reshape(-1)
        return self.intercept_ + self.coef_[0] * x
sklearn = types.ModuleType("sklearn")
linear_model = types.ModuleType("sklearn.linear_model")
linear_model.LinearRegression = LinearRegression
sys.modules.setdefault("sklearn", sklearn)
sys.modules.setdefault("sklearn.linear_model", linear_model)
print("sklearn LinearRegression fallback injected for validated additive execution")


# Week 11-1 · TBP-04 — From a Trading Idea to a Trading Robot (Backtest & Live)
### Building algo strategies, the platform choices, and a linear-regression "swing" model
*EPAT · Trading & Backtesting Platforms · executed practice notebook (instructor: Dr. Hui Liu)*

**The big picture.** A trading *strategy* is only one part of a trading *robot*. This session walks the full road —
fund setup, the menu of Python platforms, **backward vs forward testing**, the **trend-vs-swing** mindset — and
then builds a concrete **machine-learning (linear regression) swing model** that predicts tomorrow's move from
today's, backtests it, and judges it **honestly**.

> The lecture pulls SPY/QQQ/AAPL/GOOG live from a broker. That isn't available offline, so we reproduce the exact
> feature-engineering, linear-regression model, quadrant chart and swing backtest on a shipped daily index series
> (`spy_like_daily.csv`, NIFTY 50). The method and the punchline (a weak, negative "swing-world" coefficient) are
> identical.

## 1. The road map (memorise this)
Every robot follows the same path:

**idea → collect data → build a math model (prove/disprove) → visualise → program it → backtest → forward test →
live (small) → live (full)**

A *strategy* is just the idea; a *robot* wraps it with data handling, execution, risk and monitoring.

## 2. Why automate at all? (the honest reason)
Not for *more* profit — there's no guarantee of that. The real wins of algo trading are:
- **Emotionless** execution → **fewer human errors**.
- **Less stress** — no staring at 8 screens from open to close.
- You **trust your deployed robot**, freeing time for *research* (the part that actually finds edge).

> 💡 On AI bots vs hiring a coder: don't over-trust AI — you must be **knowledgeable enough to verify its output**.
> Coding is a professional job; spend *your* time on strategy research.

## 3. Running a fund: two account types
| Account | Owner | Your power | When |
|---|---|---|---|
| **Advisor** | the **client** | **trade only** (can't move money) | early — builds trust; IB sets up the legal contract |
| **Pooled** | **you / your firm** | full authority (like a mutual fund) | senior — more regulations, client must consent |

The cycle: research → translate to Python → server (local/cloud) → summarise results frequently → improve →
build a track record → manage client portfolios (mind the regulations).

## 4. The five categories of Python trading tools
You'll be flooded with options; here's the map (and the trade-offs):

| Category | Examples | Pros | Cons |
|---|---|---|---|
| **Web-based platform** | Quantopian-style | nothing to install, huge data, instant backtest, great analytics | ⚠️ **no privacy** (you hand over broker login), can't install custom packages, limited products |
| **Low-level wrappers** | `ibpy`, `ib_insync` | quick live trading | must be a pro dev, **no backtesting**, forum-only support |
| **Backtesting engines** | `zipline` | strong community, good backtests | **not for live trading**, limited products (research only) |
| **Broker native API** | IB API | can live trade, broker support | hard to build real robots, **no backtesting**, long to code |
| **Self-hosted** | the instructor's platform | **privacy**, easy, **backtest + live in ONE platform, no code change** | just software (needs a data provider), must install locally |

> 🔑 The critical requirement: a platform that does **backtest AND live trading with no code change** — changing
> code between the two is where human errors creep in.

## 5. Backward vs forward testing
- **Backward testing** — run the robot on **historical data** (like research).
- **Forward testing** — run on a **paper account** going forward, as if real.

> ⚠️ Paper-account performance ≠ live performance. Even a perfect paper result must be confirmed **live with small
> money** before scaling up.

**Privacy of a self-hosted setup:** your computer ↔ **IB Gateway/TWS** ↔ IB server. You type your broker
id/password **into IB's own software**; your strategy code (just Python files in your folder) never sees it.

## 6. The platform's skeleton
Two core functions + three cornerstones — enough to build complex robots:

```python
def initialize(context):     # runs ONCE — setup
    context.security = "SPY"

def handle_data(context):     # runs EVERY second — your logic lives here
    px   = get_realtime_price(context.security)   # cornerstone 1
    hist = get_historical_data(context.security)  # cornerstone 2
    place_order(context.security, qty)            # cornerstone 3
```
We mimic these below with plain functions on our CSV so the ideas run offline.

## 7. Imports & data — our SPY stand-in

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

df = pd.read_csv("spy_like_daily.csv", parse_dates=["Date"]).set_index("Date").sort_index()
print("rows:", len(df), "| from", df.index.min().date(), "to", df.index.max().date())
df[["Close"]].tail()

## 8. Trend vs Swing — the core mindset
- **Trend** (momentum): if it went up, you bet it keeps going up. *Buy-and-hold is the ultimate trend bet.*
  Analogy: a long friendship — you expect it to persist.
- **Swing** (mean-reversion): if it went up *too much*, you bet it falls back.

Our example strategy is **swing**: *if price dropped vs yesterday → BUY; if it rose vs yesterday → SHORT; trade
daily* (buy low, sell high).

## 9. Feature engineering by shifting (the crucial step)
Machine-learning models work **row by row**, so we line up everything we need on the same row:
- `close_yest` = `Close.shift(1)` (bring yesterday's close down a row),
- `chg_y2t` = today's % change = `(Close − close_yest) / close_yest` — the **predictor**,
- `chg_t2m` = tomorrow's % change = `chg_y2t.shift(-1)` (bring tomorrow's change up) — the **target**.

In [ ]:
d = df.copy()
d["close_yest"] = d["Close"].shift(1)
d["chg_y2t"] = (d["Close"] - d["close_yest"]) / d["close_yest"]   # predictor: yesterday->today
d["chg_t2m"] = d["chg_y2t"].shift(-1)                              # target: today->tomorrow
d = d.dropna()
d[["Close", "close_yest", "chg_y2t", "chg_t2m"]].head()

## 10. Fit the linear regression
`X = chg_y2t` (today's move) predicts `y = chg_t2m` (tomorrow's move). The **sign of the coefficient** tells us trend vs swing; its size tells us how weak the signal is.

In [ ]:
X = d[["chg_y2t"]].values
y = d["chg_t2m"].values
lr = LinearRegression().fit(X, y)
coef, intercept = lr.coef_[0], lr.intercept_
corr = np.corrcoef(d["chg_y2t"], d["chg_t2m"])[0, 1]
print("coefficient: %+.4f" % coef)
print("intercept:   %+.6f" % intercept)
print("correlation: %+.4f" % corr)
print("-> NEGATIVE coef = 'swing world' (up today -> down tomorrow)" if coef < 0
      else "-> POSITIVE coef = 'trend world'")
print("-> |corr|=%.2f is %s (statisticians want > 0.6-0.7)" % (abs(corr),
      "VERY WEAK" if abs(corr) < 0.3 else "moderate"))

## 11. The quadrant picture
Each dot is one day: x = today's move, y = tomorrow's move. Add zero lines → four quadrants. A **down-sloping**
fit line means **swing** (positive x tends to give negative y).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.scatter(d["chg_y2t"]*100, d["chg_t2m"]*100, s=8, alpha=0.35, color="#57534e")
xs = np.linspace(d["chg_y2t"].min(), d["chg_y2t"].max(), 100)
ax.plot(xs*100, (lr.predict(xs.reshape(-1,1)))*100, color="#0ea5e9", lw=2.2,
        label="fit (coef %+.3f)" % coef)
ax.axhline(0, color="k", lw=0.8); ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("% change  yesterday -> today  (predictor)")
ax.set_ylabel("% change  today -> tomorrow  (target)")
ax.set_title("Swing world: down-sloping fit (up today tends to fall tomorrow)")
ax.legend(); plt.tight_layout(); plt.show()

## 12. Backtest the swing strategy
Swing rule: **position = −sign(today's move)** — short after an up day, long after a down day. Hold one day,
realise tomorrow's return. Compare to **buy-and-hold** (the ultimate *trend* strategy).

In [ ]:
d["signal"] = -np.sign(d["chg_y2t"])          # swing: opposite of today's move
d["strat"]  = d["signal"] * d["chg_t2m"]       # earn tomorrow's return
swing = (1 + d["strat"]).cumprod()
bh    = (1 + d["chg_t2m"]).cumprod()

def stats(ret, ann=252):
    cur = (1+ret).cumprod(); tot = cur.iloc[-1]-1
    sharpe = ret.mean()/ret.std()*np.sqrt(ann)
    dd = (cur/cur.cummax()-1).min()
    return tot, sharpe, dd
s_tot, s_sh, s_dd = stats(d["strat"]); b_tot, b_sh, b_dd = stats(d["chg_t2m"])
win = (d["strat"] > 0).mean()
print("                 SWING        Buy&Hold")
print("Total return   %+7.1f%%   %+7.1f%%" % (100*s_tot, 100*b_tot))
print("Sharpe         %7.2f    %7.2f" % (s_sh, b_sh))
print("Max drawdown   %+7.1f%%   %+7.1f%%" % (100*s_dd, 100*b_dd))
print("Win rate       %7.1f%%" % (100*win))

In [ ]:
plt.figure(figsize=(11,4))
plt.plot(swing.index, swing, label="Swing (mean-reversion)", color="#0ea5e9")
plt.plot(bh.index, bh, label="Buy & hold (trend)", color="#78716c")
plt.axhline(1, color="k", lw=0.7); plt.legend()
plt.title("Swing strategy vs buy & hold (growth of 1)")
plt.tight_layout(); plt.show()

## 13. Is there really an edge? (be honest)
Test the model across a small "universe" by fitting the same regression on each series. The lecture found
SPY≈0.1, QQQ≈0.09, AAPL≈0.08, GOOG≈0.02–0.04 — all **weak negative** ("swing world", but barely). We echo that on
NIFTY plus resampled views.

In [ ]:
def coef_of(series):
    s = series.dropna()
    chg = s.pct_change()
    Xx = chg.shift(0); yy = chg.shift(-1)
    m = pd.concat([Xx, yy], axis=1).dropna()
    lr2 = LinearRegression().fit(m.iloc[:,[0]], m.iloc[:,1])
    c = np.corrcoef(m.iloc[:,0], m.iloc[:,1])[0,1]
    return lr2.coef_[0], c

rows = []
rows.append(("NIFTY daily",  *coef_of(df["Close"])))
rows.append(("NIFTY 2-day",  *coef_of(df["Close"].iloc[::2])))
rows.append(("NIFTY weekly", *coef_of(df["Close"].resample("W").last())))
print("%-14s %10s %12s" % ("series", "coef", "corr"))
for n,c,r in rows:
    print("%-14s %+10.4f %+12.4f" % (n, c, r))
print("\n-> every |coef| < 0.15: barely any signal, and the sign even flips across")
print("   timeframes (daily swing, but 2-day/weekly drift positive). The edge is thin.")

## 14. Composite figure for the study pack

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(d["chg_y2t"]*100, d["chg_t2m"]*100, s=7, alpha=0.3, color="#57534e")
ax[0].plot(xs*100, lr.predict(xs.reshape(-1,1))*100, color="#0ea5e9", lw=2.2)
ax[0].axhline(0, color="k", lw=0.8); ax[0].axvline(0, color="k", lw=0.8)
ax[0].set_xlabel("yesterday->today %"); ax[0].set_ylabel("today->tomorrow %")
ax[0].set_title("1. Swing world (coef %+.3f, corr %+.2f)" % (coef, corr), fontsize=10)
ax[1].plot(swing.index, swing, label="Swing", color="#0ea5e9")
ax[1].plot(bh.index, bh, label="Buy & hold", color="#78716c")
ax[1].axhline(1, color="k", lw=0.7); ax[1].legend()
ax[1].set_title("2. Swing vs buy & hold", fontsize=10)
plt.tight_layout(); plt.savefig("chart_1_swing.png", dpi=110, bbox_inches="tight"); plt.show()
print("saved chart_1_swing.png")

## Takeaways
1. **Robot = strategy + everything else.** Follow the road map: idea → data → model → visualise → program →
   backtest → forward test → live (small → full).
2. **Automate for discipline, not magic** — emotionless, fewer errors, less stress; spend your time on research.
3. **Pick the right platform**: privacy + **backtest & live in one, no code change** is the gold requirement; web
   platforms cost you privacy, wrappers/native-APIs lack backtesting, engines lack live.
4. **Trend vs Swing** is the core mental model; buy-and-hold is the ultimate trend bet.
5. **Feature engineering by shifting** lines up predictor (today's move) and target (tomorrow's move) on one row.
6. **Honest result:** the linear-regression coefficient is **negative** (a "swing world") but the correlation is
   **~0.1 — very weak**. The structure is real; the edge is thin. *Verify live before risking real capital.*

# Additive validation layer

These cells are appended only in this validated copy.

In [ ]:
import pandas as pd
from pathlib import Path
base = Path('.')
files = ['tbp04_html_audit.csv', 'tbp04_source_inventory.csv', 'tbp04_dependency_execution.csv', 'tbp04_regression_metrics.csv', 'tbp04_strategy_metrics.csv', 'tbp04_timeframe_controls.csv', 'tbp04_validation_controls.csv']
data = {f: pd.read_csv(base / f) for f in files}
{k: v.shape for k, v in data.items()}

## 1. Dependency and source audit

In [ ]:
print(data['tbp04_source_inventory.csv'].to_string(index=False))
print(data['tbp04_dependency_execution.csv'].to_string(index=False))
dep = data['tbp04_dependency_execution.csv'].set_index('metric')
assert dep.loc['sklearn_available','value'] in ['yes','no']

## 2. Regression and strategy metrics

In [ ]:
print(data['tbp04_regression_metrics.csv'].to_string(index=False))
print(data['tbp04_strategy_metrics.csv'].to_string(index=False))
reg = data['tbp04_regression_metrics.csv'].set_index('metric')
assert abs(float(reg.loc['correlation','value'])) < 0.1

## 3. Timeframe and scope controls

In [ ]:
print(data['tbp04_timeframe_controls.csv'].to_string(index=False))
print(data['tbp04_validation_controls.csv'].to_string(index=False))
assert data['tbp04_timeframe_controls.csv'].shape[0] == 3